# 1: Cài đặt thư viện
- Sử dụng python 3.11 (bản tháng 07/2025)

In [1]:

!pip install lightfm pyarrow fastparquet tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.1 MB/s eta 0:00:00
  Created wheel for lightfm: filename=lightfm-1.17-cp311-cp311-linux_x86_64.whl size=825355 sha256=ead94b51a04992cd7f75017492024316d03f7cb179e686226b3911a2f51f79cd
  Stored in directory: /root/.cache/pip/wheels/b9/0d/8a/0729d2e6e3ca2a898ba55201f905da7db3f838a33df5b3fcdd
Successfully built lightfm


# 2: Import thư viện, khai báo đường dẫn


In [2]:

import numpy as np
import pandas as pd

from scipy.sparse import coo_matrix
from lightfm import LightFM
from tqdm.auto import tqdm

# Nếu 3 file nằm cùng thư mục với notebook thì giữ nguyên
TRAIN_PATH = "train_logs_program.parquet"
TEST_EPG_PATH = "test_epg_candidates.parquet"
SUB_PATH = "submission.csv"
OUT_PATH = "submission_bpr.csv"   # file submit sẽ được ghi ra


# 3: Đọc dữ liệu train, test_epg, submission


In [3]:

# Log đã join sẵn với EPG, chứa tv_show_id
logs = pd.read_parquet(TRAIN_PATH)

# EPG test – chỉ cần danh sách tv_show_id xuất hiện trong test
test_epg = pd.read_parquet(TEST_EPG_PATH)

# File submission mẫu: 2 cột user_id, tv_show_id (chuỗi "0 0 0 0 0")
submission = pd.read_csv(SUB_PATH)

print("Số dòng train_logs_program:", len(logs))
print("Số dòng test_epg_candidates:", len(test_epg))
print("Số user trong submission:", len(submission))

display(logs.head())
display(test_epg.head())
display(submission.head())


Số dòng train_logs_program: 1202549
Số dòng test_epg_candidates: 144774
Số user trong submission: 1260


,user_id,tv_show_id,vsetv_id,start_time_view,end_time_view,duration_view
0,57357915868236403,6600437,353,2020-03-09 07:39:35,2020-03-09 07:44:41,306
1,8404698046253197367,6600437,353,2020-03-09 07:40:45,2020-03-09 08:06:41,1556
2,10561746661310954575,6600437,353,2020-03-09 07:40:48,2020-03-09 07:46:54,366
3,5102444605050899938,6600437,353,2020-03-09 07:48:30,2020-03-09 08:05:55,1045
4,1725562967272176802,6600437,353,2020-03-09 07:54:15,2020-03-09 07:58:09,234


,channel_id,tv_show_id,start_ts,end_ts
0,3,20088,1595831400,1595833200
1,3,2400480,1595833200,1595833800
2,3,20088,1595833800,1595836800
3,3,2400480,1595836800,1595837400
4,3,20088,1595837400,1595840400


,user_id,tv_show_id
0,8377619604347126107,0 0 0 0 0
1,8381667675275833309,0 0 0 0 0
2,8387147770138767246,0 0 0 0 0
3,8397181578236218580,0 0 0 0 0
4,8404698046253197367,0 0 0 0 0


# 4: Tiền xử lý interactions cho BPR


In [4]:

# Giữ lại các cột cần thiết
logs_small = logs[["user_id", "tv_show_id", "duration_view"]].copy()

# Gộp lại theo (user, item) – có thể dùng tổng thời lượng xem làm trọng số,
# nhưng cho BPR thì chỉ cần biết có xem hay không là đủ.
interactions_df = (
    logs_small
    .groupby(["user_id", "tv_show_id"], as_index=False)["duration_view"]
    .sum()
)

print("Số (user, item) unique:", len(interactions_df))
interactions_df.head()


Số (user, item) unique: 627456


,user_id,tv_show_id,duration_view
0,2244466330591177,200405,8
1,2244466330591177,240081,2344
2,2244466330591177,400335,5942
3,2244466330591177,700369,643
4,2244466330591177,1000384,3401


# 5: Map user_id, tv_show_id sang chỉ số và tạo sparse matrix cho LightFM


In [5]:

# Lấy danh sách user và item
unique_users = interactions_df["user_id"].unique()
unique_items = interactions_df["tv_show_id"].unique()

print("Số user train:", len(unique_users))
print("Số item train (tv_show):", len(unique_items))

# Tạo mapping id -> index
user_id_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}
item_id_to_idx = {iid: idx for idx, iid in enumerate(unique_items)}

# Mapping ngược (nếu cần)
idx_to_item_id = {idx: iid for iid, idx in item_id_to_idx.items()}

# Gán index cho bảng interactions
interactions_df["user_idx"] = interactions_df["user_id"].map(user_id_to_idx)
interactions_df["item_idx"] = interactions_df["tv_show_id"].map(item_id_to_idx)

# Tạo sparse matrix dạng (n_users, n_items)
rows = interactions_df["user_idx"].values
cols = interactions_df["item_idx"].values
data = np.ones_like(rows, dtype=np.float32)  # BPR: chỉ cần implicit 1/0

n_users = len(unique_users)
n_items = len(unique_items)

interaction_matrix = coo_matrix(
    (data, (rows, cols)),
    shape=(n_users, n_items),
    dtype=np.float32
).tocsr()

print("Kích thước ma trận tương tác:", interaction_matrix.shape)


Số user train: 4885
Số item train (tv_show): 4160
Kích thước ma trận tương tác: (4885, 4160)


# 6: Train mô hình BPR với LightFM


In [6]:

# Tham số bạn có thể tinh chỉnh thêm
NO_COMPONENTS = 64       # số chiều latent
LEARNING_RATE = 0.05
EPOCHS = 30              # tăng lên nếu thời gian cho phép
NUM_THREADS = 4          # Colab thường dùng 2–4 tuỳ cấu hình

model = LightFM(
    loss="bpr",
    no_components=NO_COMPONENTS,
    learning_rate=LEARNING_RATE,
    random_state=42
)

model.fit(
    interaction_matrix,
    epochs=EPOCHS,
    num_threads=NUM_THREADS,
    verbose=True
)


Epoch: 100%|██████████| 30/30 [00:39<00:00,  1.32s/it]


# 7: Lấy tập candidate từ test EPG + tính độ phổ biến


In [7]:

# Danh sách candidate tv_show_id xuất hiện trong test epg
candidate_tv_shows_all = test_epg["tv_show_id"].unique()

print("Số tv_show_id trong test_epg:", len(candidate_tv_shows_all))

# Chỉ giữ những tv_show_id đã xuất hiện trong train (model mới có embedding)
candidate_tv_shows = [
    tv for tv in candidate_tv_shows_all
    if tv in item_id_to_idx
]

print("Số tv_show_id candidate có embedding (train & test giao nhau):", len(candidate_tv_shows))

# Map sang item_idx để đưa vào model.predict
candidate_item_indices = np.array(
    [item_id_to_idx[tv] for tv in candidate_tv_shows],
    dtype=np.int32
)

candidate_tv_shows = np.array(candidate_tv_shows)  # cùng thứ tự với candidate_item_indices

# Tính độ phổ biến (tổng thời lượng xem) làm baseline fallback
popularity = (
    logs_small
    .groupby("tv_show_id")["duration_view"]
    .sum()
    .sort_values(ascending=False)
)

# Top tv_show phổ biến trong toàn train
popular_tv_shows_all = popularity.index.to_numpy()

# Top phổ biến nhưng phải nằm trong candidate test
popular_candidates = [tv for tv in popular_tv_shows_all if tv in candidate_tv_shows_all]

print("Số candidate có thứ hạng theo popular:", len(popular_candidates))


Số tv_show_id trong test_epg: 6636
Số tv_show_id candidate có embedding (train & test giao nhau): 2994
Số candidate có thứ hạng theo popular: 2994


# 8: Hàm sinh top-K tv_show_id cho một user


In [8]:

K = 5

def recommend_for_user(user_id):
    """
    Trả về list K tv_show_id được recommend cho user_id.
    #1 là score cao nhất (dự đoán xem nhiều nhất).
    """
    # Nếu user không xuất hiện trong train -> fallback: top phổ biến
    if user_id not in user_id_to_idx:
        # Lấy 5 tv phổ biến nhất trong candidate test
        top5 = popular_candidates[:K]
        return top5

    uidx = user_id_to_idx[user_id]

    if len(candidate_item_indices) == 0:
        # Không có candidate nào có embedding -> fallback global popular
        top5 = popular_candidates[:K]
        return top5

    # Dự đoán score cho toàn bộ candidate của user này
    scores = model.predict(uidx, candidate_item_indices, num_threads=NUM_THREADS)

    # Sắp xếp giảm dần theo score, lấy Top-K
    topk_idx = np.argsort(-scores)[:K]

    # Lấy ra tv_show_id thực
    topk_tv = candidate_tv_shows[topk_idx]

    return topk_tv


# 9: Tạo cột tv_show_id mới với 5 tv_show_id dự đoán


In [9]:

updated_submission = submission.copy()

# Đảm bảo dùng tqdm để theo dõi tiến trình
tqdm.pandas(desc="Recommending")

def make_tv_show_string(user_id):
    topk = recommend_for_user(user_id)
    # Chuyển thành chuỗi "id1 id2 id3 id4 id5"
    topk_str = " ".join(str(int(t)) for t in topk)
    return topk_str

updated_submission["tv_show_id"] = updated_submission["user_id"].progress_apply(make_tv_show_string)

updated_submission.head()


Recommending:   0%|          | 0/1260 [00:00<?, ?it/s]

,user_id,tv_show_id
0,8377619604347126107,2400480 20088 500315 200337 2500430
1,8381667675275833309,2400480 200432 200352 6500479 240081
2,8387147770138767246,12001682 12002896 500346 500331 200352
3,8397181578236218580,6000483 6500479 200352 2400480 240081
4,8404698046253197367,6600437 700389 20088 200352 2400480


# 10: Lưu file submission mới


In [10]:

updated_submission.to_csv(OUT_PATH, index=False)
print("Đã ghi file:", OUT_PATH)

# Xem vài dòng đầu để kiểm tra
updated_submission.head()


Đã ghi file: submission_bpr.csv


,user_id,tv_show_id
0,8377619604347126107,2400480 20088 500315 200337 2500430
1,8381667675275833309,2400480 200432 200352 6500479 240081
2,8387147770138767246,12001682 12002896 500346 500331 200352
3,8397181578236218580,6000483 6500479 200352 2400480 240081
4,8404698046253197367,6600437 700389 20088 200352 2400480
